In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')
!gcloud config set project YOUR_GCP_PROJECT_ID
%cd /content/drive/MyDrive/E231project
!ls benchmarks infra scripts config.yaml
!grep -c "rows_per_partition" config.yaml federated/fedavg_coordinator.py   # both > 0 = new code

In [ ]:
# Inputs from the previous session must exist: canonical training data and the sweep subsets
!gcloud storage ls gs://YOUR_BUCKET/full/processed/
!gcloud storage ls gs://YOUR_BUCKET/full/scaling_scratch/subsets/

In [ ]:
# OPTIONAL (~2 min, no cluster): federated code still matches the NumPy reference -> must print ALL PASSED
!pip -q install pyspark==3.5.3
!python tests/test_federated_reference.py 2>/dev/null | grep -E "PASS|FAIL|ALL"

In [ ]:
# Plan check: repetitions and the job settings (no cluster needed)
!grep -E "repetitions|executor_memory" config.yaml
!python scripts/run_benchmarks.py --config config.yaml --modes federated --skip-prepare --dry-run | grep -E "timed runs|run 1/"

In [ ]:
!CONFIG=config.yaml bash infra/dataproc_launch.sh 16

In [ ]:
# Federated sweeps only. Subsets already exist, so --skip-prepare. Each run now prints one line per round.
!python scripts/run_benchmarks.py --config config.yaml --modes federated --skip-prepare

In [ ]:
!CONFIG=config.yaml bash infra/dataproc_delete.sh